# 🚀 Grad-Refactor-AI: Qwen2.5-Coder-3B LoRA Training on Google Colab (Free T4 GPU)

**GPU**: Tesla T4 (Select: *Runtime* -> *Change runtime type* -> **T4 GPU**)  
**Model**: `Qwen/Qwen2.5-Coder-3B-Instruct` in Native FP16  
**LoRA Targets**: All 7 projection layers (`q, k, v, o, gate, up, down_proj`)  
**Dataset**: 799 records (guard_clauses v1)

In [ ]:
# Step 1: Install Dependencies & Verify T4 GPU
!pip install -q "transformers>=4.45.0" "peft>=0.13.0" "accelerate>=0.34.0" "datasets>=3.0.0"

import torch
assert torch.cuda.is_available(), "Please enable GPU in Runtime -> Change runtime type -> T4 GPU!"
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 2: Upload or Download records.jsonl
from pathlib import Path
from google.colab import files

DATASET_PATH = Path("records.jsonl")

if not DATASET_PATH.exists():
    print("Please upload your 'records.jsonl' dataset (3.5 MB):")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".jsonl"): 
            DATASET_PATH = Path(filename)
            break

print(f"Using dataset: {DATASET_PATH} ({DATASET_PATH.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Step 3: Load Qwen2.5-Coder-3B-Instruct & Configure LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

print(f"Loading {MODEL_ID} in native torch.float16...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
print(f"Allocated VRAM: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
# Step 4: Tokenize Guard Clauses Dataset (refactor-v2)
import json
from datasets import Dataset

train_rows = []
val_rows = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if row["split"] == "train":
            train_rows.append(row)
        elif row["split"] == "validation":
            val_rows.append(row)

print(f"Loaded {len(train_rows)} train and {len(val_rows)} validation records.")

def encode_record(row):
    prompt_text = row["prompt"]
    completion_text = row["completion"] + "<|im_end|>"
    full_text = f"<|im_start|>system\nYou are a precise Python refactoring assistant.<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n{completion_text}"
    
    prompt_ids = tokenizer.encode(f"<|im_start|>system\nYou are a precise Python refactoring assistant.<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n", add_special_tokens=False)
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids[:2048],
        "attention_mask": [1] * len(full_ids[:2048]),
        "labels": labels[:2048]
    }

train_dataset = Dataset.from_list([encode_record(r) for r in train_rows])
val_dataset = Dataset.from_list([encode_record(r) for r in val_rows])
print("Tokenization complete!")

In [ ]:
# Step 5: Training Execution
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from pathlib import Path

OUTPUT_DIR = Path("./runs/qwen3b-lora")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_steps=12,
    weight_decay=0.01,
    fp16=True,
    bf16=False,
    optim="adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt"),
)

print("Starting training on Colab T4 GPU...")
train_result = trainer.train()
print("Training finished successfully!")
print(f"Final train loss: {train_result.training_loss:.4f}")

In [ ]:
# Step 6: Save & Auto-Download Adapter
import shutil
from google.colab import files

ADAPTER_DIR = Path("./qwen3b-adapter")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

archive_name = shutil.make_archive("qwen3b-lora-adapter", "zip", ADAPTER_DIR)
print(f"Adapter archived: {archive_name}")

print("Downloading adapter to your local machine...")
files.download(archive_name)